In [ ]:
# ============================================================================
# PART 0 — SETUP & DATA
# ============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import shap

stock_dortmund = pd.read_csv("Borussia Dortmund Stock Price History.csv")
stock_dortmund["Date"] = pd.to_datetime(stock_dortmund["Date"])

match_dortmund = pd.read_csv("dortmund_2000_to_2025.csv")
match_dortmund = match_dortmund[["hometeam", "awayteam", "homeelo", "awayelo", "ftresult", "matchdate"]]
match_dortmund["Date"] = pd.to_datetime(match_dortmund["matchdate"])

df_filtered = pd.DataFrame()
df_filtered['Date'] = match_dortmund['Date']

dortmund_result = []
for i in range(len(match_dortmund)):
    if (match_dortmund['ftresult'].iloc[i] == 'A' and match_dortmund['hometeam'].iloc[i] == 'Dortmund') or \
       (match_dortmund['ftresult'].iloc[i] == 'H' and match_dortmund['awayteam'].iloc[i] == 'Dortmund'):
        dortmund_result.append(0)
    elif match_dortmund['ftresult'].iloc[i] == 'D':
        dortmund_result.append(1)
    else:
        dortmund_result.append(2)
df_filtered['dortmund_result'] = dortmund_result

dortmund_elo = []
for i in range(len(match_dortmund)):
    if match_dortmund['hometeam'].iloc[i] == 'Dortmund':
        dortmund_elo.append(match_dortmund['homeelo'].iloc[i])
    else:
        dortmund_elo.append(match_dortmund['awayelo'].iloc[i])
df_filtered['dortmund_elo'] = dortmund_elo

df_filtered['Date'] = pd.to_datetime(df_filtered['Date']).dt.date
stock_dortmund['Date'] = pd.to_datetime(stock_dortmund['Date']).dt.date
min_date = stock_dortmund['Date'].min()
max_date = max(df_filtered['Date'].max(), stock_dortmund['Date'].max())
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
combined_df = pd.DataFrame({'Date': all_dates.date})
combined_df = combined_df.merge(df_filtered, on='Date', how='left')
combined_df = combined_df.merge(stock_dortmund, on='Date', how='left')
combined_df['Price'] = combined_df['Price'].ffill()

df_clean = combined_df.dropna(subset=['dortmund_result']).reset_index(drop=True)
df_clean = df_clean.sort_values('Date').reset_index(drop=True)

# ============================================================================
# PART 1 — RANDOM FOREST (price target)
# ============================================================================
lag_columns = []
n_lags = 4
for i in range(1, n_lags + 1):
    df_clean[f'result_lag{i}'] = df_clean['dortmund_result'].shift(i)
    lag_columns.append(f'result_lag{i}')
for i in range(1, n_lags + 1):
    df_clean[f'elo_lag{i}'] = df_clean['dortmund_elo'].shift(i)
    lag_columns.append(f'elo_lag{i}')
for i in range(1, n_lags + 1):
    df_clean[f'price_lag{i}'] = df_clean['Price'].shift(i)
    lag_columns.append(f'price_lag{i}')

df_model = df_clean.dropna(subset=lag_columns).reset_index(drop=True)

X = df_model[lag_columns]
y = df_model['Price']

train_size = int(0.8 * len(X))
X_train_rf = X.iloc[:train_size]
X_test_rf  = X.iloc[train_size:]
y_train_rf = y.iloc[:train_size]
y_test_rf  = y.iloc[train_size:]

param_grid_rf = {
    'n_estimators': [100, 500],
    'max_features': list(range(2, X_train_rf.shape[1], 2)),
    'min_samples_split': list(range(20, 100, 10))
}
tscv = TimeSeriesSplit(n_splits=10)
grid_search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf, cv=tscv, scoring='neg_mean_absolute_error',
    n_jobs=-1, verbose=0, return_train_score=True
)
grid_search_rf.fit(X_train_rf, y_train_rf)

final_rf = RandomForestRegressor(**grid_search_rf.best_params_, random_state=42, n_jobs=-1)
final_rf.fit(X_train_rf, y_train_rf)
rf_test_pred = final_rf.predict(X_test_rf)
print(f"RF test MAE: {mean_absolute_error(y_test_rf, rf_test_pred):.4f}")

# ============================================================================
# PART 2 — LSTM (price target)
# LSTM_BEST_PARAMS: replace with YOUR actual grid-search winner if different.
# ============================================================================
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam

feature_columns = ['Price', 'dortmund_result', 'dortmund_elo']
df_features = df_clean[feature_columns].dropna().copy()
if len(df_features) > 400:
    df_features = df_features.drop(index=df_features.index[400]).reset_index(drop=True)

scaler_X = MinMaxScaler()
df_features_scaled = pd.DataFrame(scaler_X.fit_transform(df_features), columns=feature_columns)

window_size = 4
X_windows, y_values = [], []
for i in range(len(df_features_scaled) - window_size):
    X_windows.append(df_features_scaled.iloc[i:i+window_size].values)
    y_values.append(df_features.iloc[i+window_size]['Price'])

X_lstm = np.array(X_windows)
y_lstm = np.array(y_values).reshape(-1, 1)

train_size_lstm = int(0.8 * len(X_lstm))
X_train_lstm, X_test_lstm = X_lstm[:train_size_lstm], X_lstm[train_size_lstm:]
y_train_lstm, y_test_lstm = y_lstm[:train_size_lstm], y_lstm[train_size_lstm:]

LSTM_BEST_PARAMS = dict(architecture_type='bidirectional', num_layers=2, units=64,
                         learning_rate=0.001, dropout_rate=0.1)

def build_lstm_model(architecture_type, num_layers, units, learning_rate, dropout_rate, input_shape):
    model = Sequential()
    if architecture_type == 'bidirectional':
        model.add(Bidirectional(LSTM(units, return_sequences=(num_layers > 1)), input_shape=input_shape))
    else:
        model.add(LSTM(units, return_sequences=(num_layers > 1), input_shape=input_shape))
    model.add(Dropout(dropout_rate))
    for l in range(1, num_layers):
        model.add(LSTM(units, return_sequences=(l < num_layers - 1)))
        model.add(Dropout(dropout_rate))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss='mse')
    return model

def train_lstm(seed, X_tr=X_train_lstm, y_tr=y_train_lstm):
    import tensorflow as tf
    tf.random.set_seed(seed)
    np.random.seed(seed)
    m = build_lstm_model(**LSTM_BEST_PARAMS, input_shape=(X_tr.shape[1], X_tr.shape[2]))
    m.fit(X_tr, y_tr, epochs=100, batch_size=32, verbose=0)
    return m

final_lstm = train_lstm(seed=42)
lstm_test_pred = final_lstm.predict(X_test_lstm, verbose=0).flatten()
print(f"LSTM test MAE: {mean_absolute_error(y_test_lstm, lstm_test_pred):.4f}")

# ============================================================================
# PART 3 — RANDOM FOREST: SHAP + STABILITY + GROUPED PERMUTATION IMPORTANCE
# ============================================================================
FEATURE_GROUPS_RF = {
    'Price lags':  [c for c in lag_columns if c.startswith('price_lag')],
    'Elo lags':    [c for c in lag_columns if c.startswith('elo_lag')],
    'Result lags': [c for c in lag_columns if c.startswith('result_lag')],
}

print("="*78)
print("RANDOM FOREST — SHAP ATTRIBUTION")
print("="*78)
print("Units: SHAP values are in the same units as the model's output —")
print("        the modeled stock price level, not a probability or %.")
print(f"Background: interventional perturbation against the full training")
print(f"            set ({len(X_train_rf)} observations).")

explainer_rf = shap.TreeExplainer(final_rf, data=X_train_rf, feature_perturbation='interventional')
shap_values_rf = explainer_rf(X_test_rf)

mean_abs_shap_rf = pd.Series(np.abs(shap_values_rf.values).mean(axis=0), index=lag_columns)
group_shap_rf = pd.Series({g: mean_abs_shap_rf[cols].sum() for g, cols in FEATURE_GROUPS_RF.items()})
print("\nMean |SHAP| by group (sum of each group's feature-level attributions):")
print(group_shap_rf.sort_values(ascending=False).round(4).to_string())
print("\nNote: this is model-attributed contribution, not a significance test —")
print("       SHAP produces no p-value and says nothing about the wider population.")

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_values_rf, show=False)
plt.title("RF SHAP Attribution (small model-attributed contribution ≠ insignificance)")
plt.tight_layout()
plt.show()

def rf_shap_ranking(seed, X_tr, y_tr, X_te):
    m = RandomForestRegressor(**grid_search_rf.best_params_, random_state=seed, n_jobs=-1)
    m.fit(X_tr, y_tr)
    ex = shap.TreeExplainer(m, data=X_tr, feature_perturbation='interventional')
    sv = ex(X_te)
    return pd.Series(np.abs(sv.values).mean(axis=0), index=lag_columns).rank(ascending=False)

seed_rankings = pd.DataFrame({s: rf_shap_ranking(s, X_train_rf, y_train_rf, X_test_rf) for s in [1, 2, 3, 4, 5]})
seed_corr = seed_rankings.corr(method='spearman')
n_seeds = seed_corr.shape[0]
print("\n" + "="*78)
print("RF SHAP STABILITY ACROSS RANDOM SEEDS (Spearman rank correlation)")
print("="*78)
print(f"Mean pairwise correlation: {seed_corr.values[np.triu_indices(n_seeds, k=1)].mean():.3f}")
print("(1.0 = identical feature ranking every seed, 0 = no relationship)")

fold_rankings = {}
for fold, (tri, vli) in enumerate(tscv.split(X_train_rf), 1):
    Xtr, ytr = X_train_rf.iloc[tri], y_train_rf.iloc[tri]
    Xvl = X_train_rf.iloc[vli]
    m = RandomForestRegressor(**grid_search_rf.best_params_, random_state=42, n_jobs=-1)
    m.fit(Xtr, ytr)
    ex = shap.TreeExplainer(m, data=Xtr, feature_perturbation='interventional')
    sv = ex(Xvl)
    fold_rankings[fold] = pd.Series(np.abs(sv.values).mean(axis=0), index=lag_columns).rank(ascending=False)

fold_rank_df = pd.DataFrame(fold_rankings)
fold_corr = fold_rank_df.corr(method='spearman')
n_folds = fold_corr.shape[0]
print("\n" + "="*78)
print("RF SHAP STABILITY ACROSS CV FOLDS (Spearman rank correlation)")
print("="*78)
print(f"Mean pairwise correlation: {fold_corr.values[np.triu_indices(n_folds, k=1)].mean():.3f}")

def grouped_permutation_importance(predict_fn, X, y_true, groups, n_repeats=30, seed=42):
    """
    groups: dict of {group_name: list of column names (if X is a DataFrame)
            or column indices (if X is a 2D array)}
    """
    rng = np.random.default_rng(seed)
    is_df = isinstance(X, pd.DataFrame)
    baseline_mae = mean_absolute_error(y_true, predict_fn(X))
    rows = []
    for name, cols in groups.items():
        increases = []
        for _ in range(n_repeats):
            X_perm = X.copy()
            perm_idx = rng.permutation(len(X))
            if is_df:
                X_perm[cols] = X_perm[cols].values[perm_idx]
            else:
                X_perm[:, cols] = X_perm[:, cols][perm_idx]
            increases.append(mean_absolute_error(y_true, predict_fn(X_perm)) - baseline_mae)
        rows.append({'Group': name, 'Mean MAE Increase': np.mean(increases),
                      'CI Lower': np.percentile(increases, 2.5), 'CI Upper': np.percentile(increases, 97.5)})
    return pd.DataFrame(rows).set_index('Group')

rf_grouped_perm = grouped_permutation_importance(final_rf.predict, X_test_rf, y_test_rf, FEATURE_GROUPS_RF)
print("\n" + "="*78)
print("RF GROUPED PERMUTATION IMPORTANCE (MAE increase when whole group is scrambled)")
print("="*78)
print(rf_grouped_perm.round(4).to_string())

print("\n" + "="*78)
print("RF — SHAP vs GROUPED PERMUTATION IMPORTANCE, SIDE BY SIDE")
print("="*78)
comparison_rf = pd.DataFrame({
    'Mean |SHAP|': group_shap_rf,
    'Permutation MAE Increase': rf_grouped_perm['Mean MAE Increase']
}).sort_values('Mean |SHAP|', ascending=False)
print(comparison_rf.round(4).to_string())

# ============================================================================
# PART 4 — LSTM: SHAP + STABILITY + GROUPED PERMUTATION IMPORTANCE
# ============================================================================
feature_names_lstm = [f"{feat}_t-{window_size - t}" for t in range(window_size) for feat in feature_columns]
X_train_flat_lstm = X_train_lstm.reshape(X_train_lstm.shape[0], -1)
X_test_flat_lstm  = X_test_lstm.reshape(X_test_lstm.shape[0], -1)

def lstm_predict_flat(model):
    def _predict(X_flat):
        X_3d = X_flat.reshape(-1, window_size, len(feature_columns))
        return model.predict(X_3d, verbose=0).flatten()
    return _predict

predict_final_lstm = lstm_predict_flat(final_lstm)

print("\n" + "="*78)
print("LSTM — SHAP ATTRIBUTION")
print("="*78)
print("Units: same as RF — model output units (modeled stock price level).")
background_lstm = shap.kmeans(X_train_flat_lstm, 25)
print("Background: KernelExplainer against a 25-point k-means summary of the")
print(f"            training set ({len(X_train_flat_lstm)} observations) — a full")
print("            background set is computationally impractical for a")
print("            model-agnostic explainer on a neural network.")

explainer_lstm = shap.KernelExplainer(predict_final_lstm, background_lstm)
lstm_shap_sample = X_test_flat_lstm[:60]   # KernelExplainer is slow; sample the test set
shap_values_lstm = explainer_lstm.shap_values(lstm_shap_sample, nsamples=100)

mean_abs_shap_lstm = pd.Series(np.abs(shap_values_lstm).mean(axis=0), index=feature_names_lstm)
FEATURE_GROUPS_LSTM = {feat: [f for f in feature_names_lstm if f.startswith(feat)] for feat in feature_columns}
group_shap_lstm = pd.Series({g: mean_abs_shap_lstm[cols].sum() for g, cols in FEATURE_GROUPS_LSTM.items()})
print("\nMean |SHAP| by variable (summed across the 4 lag timesteps):")
print(group_shap_lstm.sort_values(ascending=False).round(4).to_string())
print("\nNote: model-attributed contribution, not a significance test.")

# LSTM SHAP plot — a beeswarm needs an Explanation object, not just the raw
# array shap_values() returns, so wrap it manually.
lstm_explanation = shap.Explanation(
    values=shap_values_lstm,
    data=lstm_shap_sample,
    feature_names=feature_names_lstm
)

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(lstm_explanation, show=False)
plt.title("LSTM SHAP Attribution (small model-attributed contribution ≠ insignificance)")
plt.tight_layout()
plt.show()

# Grouped bar chart — easier to read than 12 individual lag-timestep features
plt.figure(figsize=(8, 5))
group_shap_lstm.sort_values().plot(kind='barh', color='steelblue', edgecolor='black')
plt.xlabel("Mean |SHAP| (summed across the 4 lag timesteps)")
plt.title("LSTM — Grouped SHAP Attribution by Variable")
plt.tight_layout()
plt.show()

# FIX from last round: permutation importance needs INTEGER column indices
# since X_test_flat_lstm is a NumPy array, not a DataFrame like RF's.
name_to_idx = {name: i for i, name in enumerate(feature_names_lstm)}
FEATURE_GROUPS_LSTM_IDX = {feat: [name_to_idx[f] for f in cols] for feat, cols in FEATURE_GROUPS_LSTM.items()}

SEEDS_FOR_STABILITY = [1, 2, 3]
seed_rankings_lstm = {}
for s in SEEDS_FOR_STABILITY:
    m = train_lstm(seed=s)
    pred_fn = lstm_predict_flat(m)
    bg = shap.kmeans(X_train_flat_lstm, 25)
    ex = shap.KernelExplainer(pred_fn, bg)
    sv = ex.shap_values(X_test_flat_lstm[:40], nsamples=100)
    ranking = pd.Series(np.abs(sv).mean(axis=0), index=feature_names_lstm)
    grouped_ranking = pd.Series({g: ranking[cols].sum() for g, cols in FEATURE_GROUPS_LSTM.items()})
    seed_rankings_lstm[s] = grouped_ranking.rank(ascending=False)

seed_rank_df_lstm = pd.DataFrame(seed_rankings_lstm)
seed_corr_lstm = seed_rank_df_lstm.corr(method='spearman')
n_seeds2 = seed_corr_lstm.shape[0]
print("\n" + "="*78)
print("LSTM SHAP STABILITY ACROSS RANDOM SEEDS (variable-level Spearman correlation)")
print("="*78)
print(f"Mean pairwise correlation: {seed_corr_lstm.values[np.triu_indices(n_seeds2, k=1)].mean():.3f}")
print("(Only 3 seeds used here — LSTM retraining + KernelExplainer is expensive.")
print(" Increase SEEDS_FOR_STABILITY if you want a tighter estimate.)")

lstm_grouped_perm = grouped_permutation_importance(
    predict_final_lstm, X_test_flat_lstm, y_test_lstm.flatten(), FEATURE_GROUPS_LSTM_IDX
)
print("\n" + "="*78)
print("LSTM GROUPED PERMUTATION IMPORTANCE (MAE increase when whole channel is scrambled)")
print("="*78)
print(lstm_grouped_perm.round(4).to_string())

print("\n" + "="*78)
print("LSTM — SHAP vs GROUPED PERMUTATION IMPORTANCE, SIDE BY SIDE")
print("="*78)
comparison_lstm = pd.DataFrame({
    'Mean |SHAP|': group_shap_lstm,
    'Permutation MAE Increase': lstm_grouped_perm['Mean MAE Increase']
}).sort_values('Mean |SHAP|', ascending=False)
print(comparison_lstm.round(4).to_string())